# Lesson 21: Diagnose demonstrations and test a data change

[Previous: Lesson 20](../20-smolvla-adaptation/20-smolvla-adaptation.ipynb) · [Next: Lesson 22](../22-comparison/22-comparison.ipynb) · [Course README](../../README.md) · [Course index](../index.ipynb)

This is the reference notebook. Its pair is [21-data-quality.ipynb](21-data-quality.ipynb). Run the cells in order. The learner copy stops at the first exercise that you have not written. The reference copy has all answers.

**Result of this lesson.** You will check that robot demonstrations pair each picture with the correct action. You will inspect the quality of demonstrations and of policy failures. Then you will change the training data in one controlled way and measure the change on states that the training never used. The comparison is controlled in compute too: both policies continue for the same number of updates from the same weights, and only the data differ.

**Time.** About two hours of reading and coding. The full experiment is separate. It has a target budget of 7200 seconds on one GPU with 48 GB of memory.

**Evidence boundary.** The small examples in this notebook use made-up numbers. They check your code. They do not show that a policy is good. The cells with an *experiment* tag use real recordings, a real checkpoint, real training, saved and reloaded checkpoints, and the real simulator. A number is a measurement only after you run those cells.

## 1. Why we check data before we train

A policy copies the actions in its training data. Suppose one action is paired with the wrong camera image. The policy then learns a wrong rule. The training loss can still go down, because the loss only measures how well the policy copies the pairs. A data fault is therefore easy to miss if you look only at the loss.

So we do three things in this order:

1. **Diagnose.** Check the timing of each recording. Reject a recording that has a defect. Do not repair it by guessing.
2. **Inspect.** Measure simple properties of the actions: idle time, extreme values, gripper switches, and sudden jumps. Look at the policy failures that you want to fix.
3. **Intervene.** Change one thing in the training data. Keep everything else the same. Measure the result on cases that the training did not use.

**Words used in this lesson.**

| Word | Meaning |
| --- | --- |
| tick | One step of the simulator. At 20 Hz, one tick is 50 ms. |
| observation `o_i` | What the robot sees at the start of tick `i`: two camera images and the robot state. |
| action `a_i` | The seven numbers that the controller applies during tick `i`. |
| transition row | One pair `(o_i, a_i)`. |
| window | `H` actions in a row, starting at row `s`. ACT trains to predict a window. |
| padding mask | One flag for each window slot. `True` means: this slot is outside the episode. |
| storage label | A frame rate in the dataset file. It does not control the simulator. |

## 2. Timing: one action for each tick

LIBERO runs the controller at **20 Hz**. One tick is `1 / 20 = 0.05 s = 50 ms`. In every tick, three things happen in this order:

1. The simulator gives the observation `o_i`.
2. The policy (or a person) chooses the action `a_i` from `o_i`.
3. The simulator applies `a_i` for one tick and gives `o_{i+1}`.

An episode with `N` actions therefore has `N + 1` observations. The last observation `o_N` has no action after it. It is not a training row. The saved ticks are `observation_ticks = [0, 1, ..., N]` (shape `(N+1,)`) and `action_ticks` (shape `(N,)`). For a correct recording, `action_ticks[i] = observation_ticks[i] = i`.

<svg xmlns="http://www.w3.org/2000/svg" width="640" height="190" viewBox="0 0 640 190" role="img" aria-label="Timeline of observations and actions">
  <style>text{font-family:sans-serif;font-size:13px} .o{fill:#dbeafe;stroke:#1d4ed8} .a{fill:#dcfce7;stroke:#15803d} .bad{fill:#fee2e2;stroke:#b91c1c}</style>
  <text x="10" y="22" font-weight="bold">Correct recording (N = 3)</text>
  <circle class="o" cx="60" cy="55" r="18"/><text x="48" y="60">o0</text>
  <circle class="o" cx="180" cy="55" r="18"/><text x="168" y="60">o1</text>
  <circle class="o" cx="300" cy="55" r="18"/><text x="288" y="60">o2</text>
  <circle class="o" cx="420" cy="55" r="18"/><text x="408" y="60">o3</text>
  <rect class="a" x="40" y="85" width="40" height="24"/><text x="52" y="102">a0</text>
  <rect class="a" x="160" y="85" width="40" height="24"/><text x="172" y="102">a1</text>
  <rect class="a" x="280" y="85" width="40" height="24"/><text x="292" y="102">a2</text>
  <text x="440" y="102">o3 has no action: it is not a row</text>
  <text x="10" y="140" font-weight="bold">Defect: action tick labels are one tick late</text>
  <rect class="bad" x="40" y="150" width="40" height="24"/><text x="46" y="167">label 1</text>
  <rect class="bad" x="160" y="150" width="40" height="24"/><text x="166" y="167">label 2</text>
  <rect class="bad" x="280" y="150" width="40" height="24"/><text x="286" y="167">label 3</text>
  <text x="360" y="167">action_ticks - observation_ticks[:N] = [1]</text>
</svg>

Text version of the diagram:

```text
tick:          0      1      2      3
observation:  o0     o1     o2     o3      (N+1 = 4 values)
action:       a0     a1     a2             (N   = 3 values)
defect:     label1 label2 label3            (offset = +1 tick)
```

**Offset.** The offset of action `i` is `d_i = action_ticks[i] - observation_ticks[i]`, in ticks. Each `d_i` is an integer. For a correct recording, every `d_i` is 0. In the defect above, every `d_i` is 1. A defect of one tick is `50 ms`. That is a large error: in 50 ms the robot arm moves by an amount that matters for a grasp.

**Do not repair by guessing.** If the labels disagree, you do not know which label is wrong. Moving all actions by one row can hide the error in the middle of the episode and create a new error at the end. Reject the recording and record it again. This lesson repairs only a defect that the course created and labelled itself.

**Rates that you must not mix up.**

* The control rate is `LIBERO_CONTROL_HZ = 20`. One saved action is one 50 ms tick. A recording of `N` ticks lasts `N / 20` seconds.
* The source dataset file stores frames with timestamps at `SOURCE_STORAGE_FPS = 10`. This is a *storage label*. It selects neighbouring saved rows when LeRobot builds a window. It does not change the simulator, and you must never repeat or skip actions because of it.
* Windows use **row offsets**. A window that starts at row `s` and has `H` slots uses rows `s, s+1, ..., s+H-1`. The helper `frame_offset_seconds` shows the lookup labels for those offsets. They are labels, not elapsed simulator time.

## 3. Setup

The first code cell finds the workspace and sets offline mode. Nothing downloads in this lesson. The second code cell has all imports.

In [ ]:
import os
from pathlib import Path

WORKSPACE = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "manifests" / "act_references.yaml").is_file()
    ),
    None,
)
if WORKSPACE is None:
    raise FileNotFoundError("Start Jupyter inside the tutorial workspace.")
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"

In [ ]:
import json
import math
import tempfile
from collections.abc import Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from numpy.typing import ArrayLike

from vla_lab.contracts import ActionSchema
from vla_lab.data.recording import RecordedEpisode, apply_synthetic_timing_defect, replay_episode
from vla_lab.data.recovery_dataset import RecoveryDataset
from vla_lab.evaluation.states import evaluate_act_states
from vla_lab.experiments.collect_recovery import collect_reference_recovery
from vla_lab.experiments.recovery import load_recovery_manifest
from vla_lab.sim.episode import open_libero_task
from vla_lab.teaching.course_comparison import (
    ACT_CHUNK_SIZE,
    ActionBatch,
    ActionQuality,
    AdaptationSettings,
    aligned_transition_rows,
    case_id,
    checkpoint_heldout_loss,
    continue_act_matched,
    episode_time_summary,
    intervention_case_rows,
    limit_gpu_memory,
    max_leaf_difference,
    new_run_dir,
    read_jsonl,
    require_assets,
    sha256_file,
    tick_vectors,
    write_run_identity,
)
from vla_lab.teaching.data import action_window
from vla_lab.teaching.evaluation import development_cases
from vla_lab.teaching.interfaces import RawObservationValue
from vla_lab.teaching.records import InterventionCase, InterventionSummary, TransitionAlignment
from vla_lab.timing import LIBERO_CONTROL_HZ, SOURCE_STORAGE_FPS, frame_offset_seconds

The cell below shows the numbers from section 2 in code. `frame_offset_seconds` gives lookup labels for row offsets. They are not simulator time.

In [ ]:
steps = 400
print("control rate (Hz):", LIBERO_CONTROL_HZ, "| tick length (ms):", 1000 / LIBERO_CONTROL_HZ)
print("a recording with", steps, "actions lasts", steps / LIBERO_CONTROL_HZ, "s")
print("storage-label rate (frames per second):", SOURCE_STORAGE_FPS)
print("lookup labels for rows 0..3:", frame_offset_seconds([0, 1, 2, 3]))

## Exercise 21.1: Diagnose the tick alignment

**Purpose.** Find a timing defect before you build any training rows.

**Contract.** `transition_alignment(observation_ticks, action_ticks) -> TransitionAlignment`

**Inputs.**

* `observation_ticks`: integers, shape `(N+1,)`, `[0, 1, ..., N]`.
* `action_ticks`: integers, shape `(N,)`. `action_ticks[i]` is the tick label that was saved with action `i`.

The supplied helper `tick_vectors(observation_ticks, action_ticks)` checks the shapes and returns two integer vectors. It raises `ValueError` for a wrong shape or for observation ticks that do not start at zero and rise by one.

**Output.** A dictionary with these keys:

| Key | Value |
| --- | --- |
| `aligned` | `True` if every offset `d_i` is 0. |
| `offset_ticks` | The sorted list of different offsets, as Python `int` values, in ticks. |
| `control_hz` | `LIBERO_CONTROL_HZ` (20). |
| `source_storage_fps` | `SOURCE_STORAGE_FPS` (10). |
| `simulated_duration_s` | `N / control_hz`, in seconds. |

**Worked example.** `observation_ticks = [0, 1, 2, 3]` and `action_ticks = [1, 2, 3]`. Then `d = [1, 1, 1]`. The result is `aligned=False`, `offset_ticks=[1]`, and `simulated_duration_s = 3 / 20 = 0.15`.

**Method.** Subtract the first `N` observation ticks from the action ticks. Do not change the saved actions to hide an error.

<details><summary>Hint</summary>

`observation_ticks[:-1]` has `N` values. Subtract it from `action_ticks`. Use `set` to find the different offsets. Convert each value with `int(...)`, because NumPy integers do not serialize in JSON.

</details>

In [ ]:
def transition_alignment(
    observation_ticks: Sequence[int] | np.ndarray, action_ticks: Sequence[int] | np.ndarray
) -> TransitionAlignment:
    observations, actions = tick_vectors(observation_ticks, action_ticks)
    offsets = actions - observations[:-1]
    return {
        "aligned": bool(np.all(offsets == 0)),
        "offset_ticks": sorted({int(offset) for offset in offsets}),
        "control_hz": LIBERO_CONTROL_HZ,
        "source_storage_fps": SOURCE_STORAGE_FPS,
        "simulated_duration_s": len(actions) / LIBERO_CONTROL_HZ,
    }

### Check 21.1

In [ ]:
clean = transition_alignment([0, 1, 2, 3], [0, 1, 2])
assert clean["aligned"] is True and clean["offset_ticks"] == [0], (
    f"A clean recording was rejected: {clean}"
)
late = transition_alignment([0, 1, 2, 3], [1, 2, 3])
assert late["aligned"] is False and late["offset_ticks"] == [1], (
    f"Expected one tick offset, got {late}"
)
assert late["simulated_duration_s"] == 3 / 20, "Duration is N / 20 Hz"
mixed = transition_alignment([0, 1, 2, 3], [0, 2, 2])
assert mixed["offset_ticks"] == [0, 1], "List each different offset once, in order"
assert all(type(value) is int for value in mixed["offset_ticks"]), "Use Python int values"
for bad in (([0, 1], [0, 1]), ([1, 2, 3], [1, 2]), ([0, 2, 3], [0, 2])):
    try:
        transition_alignment(*bad)
    except ValueError:
        continue
    raise AssertionError(f"Wrong tick shapes must raise ValueError: {bad}")
print("21.1 passed:", late)

## 4. Transition rows and action windows

**Transition rows.** A *transition row* is a dictionary with three keys: `tick` (an integer), `observation` (`o_i`) and `action` (`a_i`, shape `(7,)`, `float32`). The training class `RecoveryDataset` asks for one row for every applied action, in order. It checks that row `i` has `tick == i`, that the observation is the original `o_i`, and that the action is exactly the saved `a_i`. The supplied function `aligned_transition_rows(observations, actions, observation_ticks, action_ticks)` makes these rows. It pairs observation `i` with action `i`, drops the final observation, copies every action, and raises `ValueError` if the recording has a tick offset. It is plain bookkeeping, so it is given to you. The judgement is in the diagnosis that comes before it (Exercise 21.1) and in the batch that comes after it (Exercise 21.2).

**Window.** ACT predicts a *chunk* of `H = 20` future actions from one observation. For a row at index `s`, the training target is the window `W = [a_s, a_{s+1}, ..., a_{s+H-1}]`, shape `(H, 7)`. Near the end of an episode there are fewer than `H` actions left. The window is then padded with zeros, and the padding mask `m` (shape `(H,)`) is `True` for each padded slot. The loss ignores padded slots. The supplied function `action_window(actions, start=s, horizon=H)` returns `(W, m)`. You wrote an equivalent in Lesson 3. The window is made of **row offsets** `0, 1, ..., H-1` from the start row. It never uses an action from the next episode, and it never skips rows.

**Worked example.** An episode has `N = 5` actions. Take `H = 4` and `s = 3`. Real slots: `a_3, a_4`. Padded slots: two. So `m = [False, False, True, True]` and the last two rows of `W` are zero.

In [ ]:
toy_actions = np.arange(1, 6, dtype=np.float32)[:, None] * np.ones((1, 7), dtype=np.float32)
window, mask = action_window(toy_actions, start=3, horizon=4)
print("window first column:", window[:, 0], "| padding mask:", mask)
assert window.shape == (4, 7) and mask.tolist() == [False, False, True, True]

The next cell defines two small made-up recordings that the checks and the toy training class use. The actions are random numbers. The observations hold only a tick number. They say nothing about a robot.

In [ ]:
def synthetic_episode(steps: int, *, episode_id: str, reset_seed: int) -> RecordedEpisode:
    generator = np.random.default_rng(reset_seed)
    actions = generator.uniform(-0.5, 0.5, size=(steps, 7)).astype(np.float32)
    observations = tuple({"state": np.full(8, tick, dtype=np.float32)} for tick in range(steps + 1))
    return RecordedEpisode(
        episode_id=episode_id,
        task_id="bowl_plate",
        actions=actions,
        action_ticks=np.arange(steps),
        observation_ticks=np.arange(steps + 1),
        observations=observations,
        reset_seed=reset_seed,
    )


def toy_transform(observation: Mapping[str, RawObservationValue]) -> dict[str, torch.Tensor]:
    return {
        "observation.images.image": torch.zeros(3, 256, 256),
        "observation.images.image2": torch.zeros(3, 256, 256),
        "observation.state": torch.as_tensor(observation["state"], dtype=torch.float32),
    }

## Exercise 21.2: Diagnose, then prepare a batch of action windows

**Purpose.** A batch for training must come from recordings that passed the timing diagnosis, and every window must stay inside its own episode. Put both rules in one function, so that a defective recording can never reach the batch.

**Contract.** `prepare_action_batch(episodes, picks, horizon=ACT_CHUNK_SIZE) -> ActionBatch`

**Inputs.**

* `episodes`: a sequence of `RecordedEpisode`. Each has `episode_id`, `actions` `(N, 7)`, `observation_ticks` `(N+1,)` and `action_ticks` `(N,)`.
* `picks`: a sequence of `(episode_index, row)` pairs. `row` is the start row `s` of one window.
* `horizon`: the window length `H`.

**Output.** A dictionary with `action` (`float32`, shape `(B, H, 7)`), `action_is_pad` (`bool`, shape `(B, H)`), `episode_ids` (one name for each pick) and `rows` (the start rows). `B` is the number of picks.

**Rules.**

1. Raise `ValueError` if `episodes` or `picks` is empty.
2. Run your `transition_alignment` on **every** episode first, even one that no pick uses. If one is not aligned, raise `ValueError` with its `episode_id` and offsets in the message.
3. Raise `ValueError` for an `episode_index` outside the list, or a `row` outside `0..N-1` of its episode.
4. Build each window with the supplied `action_window`. Padding marks the slots after the end of *that* episode.

**Worked example.** Episode `a` has 5 actions, episode `b` has 3. Picks `[(0, 3), (1, 1)]` with `H = 4`. Window 0 is `[a_3, a_4, 0, 0]` of episode `a`. Window 1 is `[a_1, a_2, 0, 0]` of episode `b`. Both masks are `[False, False, True, True]`. The batch has shape `(2, 4, 7)`.

<details><summary>Hint</summary>

Loop over `episodes` for the diagnosis, then loop over `picks`. `action_window(actions, start=row, horizon=horizon)` returns the window and its mask. `np.stack` joins the windows into one array.

</details>

In [ ]:
def prepare_action_batch(
    episodes: Sequence[RecordedEpisode],
    picks: Sequence[tuple[int, int]],
    horizon: int = ACT_CHUNK_SIZE,
) -> ActionBatch:
    if not episodes or not picks:
        raise ValueError("need at least one episode and one pick")
    for episode in episodes:
        report = transition_alignment(episode.observation_ticks, episode.action_ticks)
        if not report["aligned"]:
            raise ValueError(
                f"{episode.episode_id}: actions are offset by {report['offset_ticks']} control ticks"
            )
    windows, masks = [], []
    for episode_index, row in picks:
        if not 0 <= episode_index < len(episodes):
            raise ValueError(f"there is no episode {episode_index}")
        actions = episodes[episode_index].actions
        if not 0 <= row < len(actions):
            raise ValueError(
                f"row {row} is outside episode {episode_index} with {len(actions)} actions"
            )
        window, mask = action_window(actions, start=row, horizon=horizon)
        windows.append(window)
        masks.append(mask)
    return {
        "action": np.stack(windows).astype(np.float32),
        "action_is_pad": np.stack(masks).astype(np.bool_),
        "episode_ids": [episodes[index].episode_id for index, _ in picks],
        "rows": [row for _, row in picks],
    }

### Check 21.2

In [ ]:
long_episode = synthetic_episode(5, episode_id="a", reset_seed=1)
short_episode = synthetic_episode(3, episode_id="b", reset_seed=2)
batch = prepare_action_batch([long_episode, short_episode], [(0, 3), (1, 1)], horizon=4)
assert batch["action"].shape == (2, 4, 7) and batch["action"].dtype == np.float32
assert batch["action_is_pad"].shape == (2, 4) and batch["action_is_pad"].dtype == np.bool_
assert batch["action_is_pad"].tolist() == [[False, False, True, True]] * 2, (
    "Padding follows each episode's own end"
)
assert np.array_equal(batch["action"][0, :2], long_episode.actions[3:5]), (
    "Window 0 holds a_3 and a_4"
)
assert np.array_equal(batch["action"][1, :2], short_episode.actions[1:3]), (
    "Window 1 never reads episode a"
)
assert not batch["action"][:, 2:].any(), "Padded slots are zero"
assert batch["episode_ids"] == ["a", "b"] and batch["rows"] == [3, 1]
defective_episode = apply_synthetic_timing_defect(long_episode)
for bad_call in (
    lambda: prepare_action_batch([short_episode, defective_episode], [(0, 0)], horizon=4),
    lambda: prepare_action_batch([long_episode], [(1, 0)], horizon=4),
    lambda: prepare_action_batch([long_episode], [(0, 5)], horizon=4),
    lambda: prepare_action_batch([long_episode], [(0, -1)], horizon=4),
    lambda: prepare_action_batch([long_episode], [], horizon=4),
    lambda: prepare_action_batch([], [(0, 0)], horizon=4),
):
    try:
        bad_call()
    except ValueError:
        continue
    raise AssertionError("A defective recording or a bad pick must raise ValueError")
print("21.2 passed:", batch["episode_ids"], batch["rows"])

### Check your batch against the real training class

`RecoveryDataset` is the class that the real experiment uses. The cell below builds a made-up recording with 25 actions and gives the class the supplied row builder and `action_window`. A toy observation transform replaces the camera processing, so the cell runs on a CPU. Your batch function must agree with the class: the window of the same row must be identical. The cell then makes a copy of the recording with the course's labelled tick defect and shows that the row builder refuses it.

The recording uses seed `20260930` and task `bowl_plate`. These values are part of the identity checks in the class.

In [ ]:
with tempfile.TemporaryDirectory() as folder:
    toy = synthetic_episode(25, episode_id="toy-clean", reset_seed=20260930)
    toy_path = toy.save(Path(folder) / "toy-clean.npz")
    toy_dataset = RecoveryDataset(
        [toy_path],
        task_id="bowl_plate",
        expected_reset_seeds={20260930},
        observation_transform=toy_transform,
        transition_rows_fn=aligned_transition_rows,
        window_fn=action_window,
    )
    toy_sample = toy_dataset[22]
    toy_batch = prepare_action_batch([toy], [(0, 22)], horizon=ACT_CHUNK_SIZE)
    print("samples:", len(toy_dataset), "| action chunk:", tuple(toy_sample["action"].shape))
    print("mask of the sample at row 22:", toy_sample["action_is_pad"].tolist())
    assert len(toy_dataset) == 25, "One training sample for each recorded tick"
    assert toy_sample["action_is_pad"].tolist() == [False] * 3 + [True] * 17
    assert torch.equal(toy_sample["action"][0], torch.from_numpy(toy.actions[22]))
    assert np.array_equal(toy_sample["action"].numpy(), toy_batch["action"][0]), (
        "Your window equals the class window"
    )
    assert np.array_equal(toy_sample["action_is_pad"].numpy(), toy_batch["action_is_pad"][0])

    defective = apply_synthetic_timing_defect(toy)
    try:
        aligned_transition_rows(
            defective.observations,
            defective.actions,
            defective.observation_ticks,
            defective.action_ticks,
        )
    except ValueError as error:
        print("defect rejected:", error)
    else:
        raise AssertionError("A recording with a tick offset must be rejected")

## 5. Inspect the actions

A demonstration can have correct timing and still be a poor example. Four simple measurements find many problems. Let `A` be the action array, shape `(N, 7)`. Columns `0..5` are the arm commands: three translations and three rotations. Each is a unitless relative command in `[lower_j, upper_j]` (here `[-1, 1]`). Column `6` is the gripper: `-1` means open and `+1` means close.

| Measurement | Definition | What it can show |
| --- | --- | --- |
| `idle_fraction` | Fraction of ticks where `max_{j<6} abs(A[i, j]) < epsilon`. The default `epsilon` is `0.05`. | The arm waits. Long waiting teaches a policy to wait. |
| `saturated_fraction` | Fraction of the `N * 6` arm values within `edge_margin` of a bound. A value `v` counts if `v <= lower_j + m * (upper_j - lower_j)` or `v >= upper_j - m * (upper_j - lower_j)`. The default `m` is `0.01`. | The controller is pushed to its limit. The clip function may hide the real intent. |
| `gripper_switches` | The number of ticks `i >= 1` where `(A[i, 6] > 0)` differs from `(A[i-1, 6] > 0)`. | One grasp and one release is normal. Many switches suggest a failed grasp. |
| `max_step_change` | `max over i >= 1 and j < 6 of abs(A[i, j] - A[i-1, j])`. It is `0.0` when `N = 1`. | A sudden jump between two ticks. |

The report also has `steps` (`N`) and `duration_s` (`N / 20`). The parameters change the result like this: a larger `epsilon` marks more ticks as idle. A larger `edge_margin` marks more values as saturated. The gripper column is not used for idle or saturation, because it is always at a bound.

**Worked example.** Four ticks:

```text
tick 0: arm = [0.5, 0, 0, 0, 0, 0], gripper = -1
tick 1: arm = [0.5, 0, 0, 0, 0, 0], gripper = -1
tick 2: arm = [0,   0, 0, 0, 0, 0], gripper = +1
tick 3: arm = [1.0, 0, 0, 0, 0, 0], gripper = +1
```

Tick 2 is idle, so `idle_fraction = 1/4 = 0.25`. One value (`1.0`) is at the bound, so `saturated_fraction = 1/24`. The gripper changes once, so `gripper_switches = 1`. The largest jump is `abs(1.0 - 0) = 1.0`, so `max_step_change = 1.0`. `duration_s = 4 / 20 = 0.2`.

## Exercise 21.3: Measure the quality of a demonstration

**Contract.** `action_quality_report(actions, *, lower, upper, idle_threshold=0.05, edge_margin=0.01) -> ActionQuality`

`lower` and `upper` are sequences of seven bounds. Raise `ValueError` if `actions` does not have shape `(N, 7)` with `N >= 1`. Return Python `int` and `float` values.

<details><summary>Hint</summary>

Split `A[:, :6]` and `A[:, 6]`. Use `np.abs(arm).max(axis=1) < idle_threshold` for idle ticks. Use `np.diff` for the changes between ticks. For saturation, compare each column with its own bounds: arrays of shape `(6,)` broadcast against `(N, 6)`.

</details>

In [ ]:
def action_quality_report(
    actions: ArrayLike,
    *,
    lower: Sequence[float],
    upper: Sequence[float],
    idle_threshold: float = 0.05,
    edge_margin: float = 0.01,
) -> ActionQuality:
    values = np.asarray(actions, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != 7 or values.shape[0] < 1:
        raise ValueError("actions must have shape (N, 7) with N >= 1")
    arm, gripper = values[:, :6], values[:, 6]
    low = np.asarray(lower, dtype=np.float64)[:6]
    high = np.asarray(upper, dtype=np.float64)[:6]
    span = high - low
    at_edge = (arm <= low + edge_margin * span) | (arm >= high - edge_margin * span)
    changes = np.abs(np.diff(arm, axis=0))
    return {
        "steps": len(values),
        "duration_s": len(values) / LIBERO_CONTROL_HZ,
        "idle_fraction": float(np.mean(np.abs(arm).max(axis=1) < idle_threshold)),
        "saturated_fraction": float(np.mean(at_edge)),
        "gripper_switches": int(np.count_nonzero(np.diff(gripper > 0))),
        "max_step_change": float(changes.max()) if len(changes) else 0.0,
    }

### Check 21.3

In [ ]:
SCHEMA = ActionSchema.libero_relative()
worked = np.zeros((4, 7))
worked[0, 0] = worked[1, 0] = 0.5
worked[3, 0] = 1.0
worked[:2, 6] = -1.0
worked[2:, 6] = 1.0
quality = action_quality_report(worked, lower=SCHEMA.lower, upper=SCHEMA.upper)
assert quality["steps"] == 4 and math.isclose(quality["duration_s"], 0.2)
assert math.isclose(quality["idle_fraction"], 0.25), quality
assert math.isclose(quality["saturated_fraction"], 1 / 24), quality
assert quality["gripper_switches"] == 1 and math.isclose(quality["max_step_change"], 1.0), quality
single = action_quality_report(np.zeros((1, 7)), lower=SCHEMA.lower, upper=SCHEMA.upper)
assert single["max_step_change"] == 0.0 and single["idle_fraction"] == 1.0
loose = action_quality_report(worked, lower=SCHEMA.lower, upper=SCHEMA.upper, idle_threshold=0.6)
assert math.isclose(loose["idle_fraction"], 0.75), (
    "A larger idle threshold marks more ticks as idle"
)
for bad_shape in (np.zeros((0, 7)), np.zeros((3, 6)), np.zeros(7)):
    try:
        action_quality_report(bad_shape, lower=SCHEMA.lower, upper=SCHEMA.upper)
    except ValueError:
        continue
    raise AssertionError("A wrong action shape must raise ValueError")
print("21.3 passed:", quality)

## 6. Design a controlled data intervention

An *intervention* changes one input and measures the output. In this lesson the input is the training data of an ACT policy. The output is the success of the policy in the simulator.

**What stays fixed.** The starting weights, the optimizer type and its settings, the random seed, the batch size, the number of updates, the evaluation states, the reset seeds, the simulator, the evaluation code, and the time limit of 300 ticks (15 s). **Only the data differ.**

**Why the baseline also trains.** Suppose you compare the original checkpoint with a checkpoint that trained 300 more updates on new recordings. Then two things differ: the data *and* the extra compute. A policy that trains longer can improve on the old data alone. You could not tell which one helped. So the baseline here is the same starting checkpoint **continued for the same 300 updates on the original training data**. The intervention continues for the same 300 updates on the original data **plus** the new recordings. Both runs start from the same weights, each with a *new* AdamW state, with the same seed and the same batch size. The original checkpoint is evaluated too, but only as context: it shows what 300 more updates do by themselves.

**State discipline.** Each simulator start state has an ID. The course splits the first twenty IDs by purpose, and the supplied function `development_cases(stage)` returns them:

| Stage | States | Use |
| --- | --- | --- |
| `exploration` | 0–9 | Collect and inspect demonstrations. Try ideas. |
| `method` | 10–14 | Choose between methods. This lesson does not use them. |
| `intervention` | 15–19 | Measure the data change. Never train on these. |

Each state ID is paired with a reset seed: `reset_seed = 20260930 + state_id`. A *case* is one state and its seed. The case name is `state-<id>-seed-<seed>`.

**Where the data comes from.** The real experiment asks the starting ACT policy to attempt exploration states. It saves each attempt as a recording. It keeps only the attempts that reached the task goal (accepted) and keeps the others apart (rejected). Training on the accepted attempts is *filtered behaviour cloning*: the policy copies its own successes. The data have a clear provenance limit: they are policy-generated, not human. A person can record better data with the keyboard. See [the recording guide](../../docs/recording.md) and the optional extensions at the end.

<svg xmlns="http://www.w3.org/2000/svg" width="660" height="120" viewBox="0 0 660 120" role="img" aria-label="Pipeline of the controlled data intervention">
  <style>text{font-family:sans-serif;font-size:12px} rect{fill:#eef2ff;stroke:#4338ca} .e{fill:#fef9c3;stroke:#a16207}</style>
  <rect x="5" y="30" width="105" height="50"/><text x="14" y="52">Start ACT</text><text x="14" y="68">checkpoint</text>
  <rect x="135" y="30" width="105" height="50"/><text x="144" y="52">Attempt states</text><text x="144" y="68">0–9, record</text>
  <rect x="265" y="30" width="105" height="50"/><text x="274" y="52">Diagnose and</text><text x="274" y="68">keep accepted</text>
  <rect x="395" y="30" width="105" height="50"/><text x="404" y="52">Two arms, same</text><text x="404" y="68">300 updates</text>
  <rect class="e" x="525" y="30" width="130" height="50"/><text x="534" y="52">Reload, evaluate</text><text x="534" y="68">states 15–19, paired</text>
  <path d="M110 55h25M240 55h25M370 55h25M500 55h25" stroke="#4338ca" fill="none"/>
</svg>

```text
start -> attempt states 0-9 -> diagnose, keep accepted -> arm A: 300 updates on original data / arm B: 300 updates on original + new data -> save, reload, evaluate both on 15-19
```

**Paired outcomes.** For each case there is one baseline-arm outcome (original data only) and one intervention-arm outcome (original plus new data). Each outcome is success (S) or failure (F). A *gain* is F then S. A *loss* is S then F. A *tie* is the same result twice. Every attempt stays in the count. A failed attempt, a timeout, and an error all count as failure.

**Worked example.** Five cases:

| Case | state 15 | 16 | 17 | 18 | 19 |
| --- | --- | --- | --- | --- | --- |
| baseline arm | F | S | F | F | S |
| intervention arm | S | S | F | F | F |

Case 15 is a gain. Case 19 is a loss. The others are ties. The success count is `2` before and `2` after. The net change is zero.

**How much can five cases show?** Under "the change has no effect", a case that is not a tie is a gain or a loss with equal chance `1/2`. Let `g` be the gains, `l` the losses, and `n = g + l`. The exact two-sided sign-test value is `p = min(1, 2 * P(X <= min(g, l)))`, where `X` counts successes in `n` fair coin flips: `P(X <= k) = sum_{i=0..k} C(n, i) / 2^n`. For `g = 2, l = 0`: `n = 2`, `P(X <= 0) = 1/4`, `p = 0.5`. The best possible result with five cases is `g = 5, l = 0`: `p = 2 / 32 = 0.0625`. That is *above* 0.05. Five cases cannot give a strong result. Treat the outcome as a pilot signal and report the counts.

In [ ]:
def sign_test_p(gains: int, losses: int) -> float:
    n = gains + losses
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(min(gains, losses) + 1)) / 2**n
    return min(1.0, 2 * tail)


print("g=2, l=0 ->", sign_test_p(2, 0))
print("g=5, l=0 ->", sign_test_p(5, 0))
print("g=1, l=1 ->", sign_test_p(1, 1))
assert sign_test_p(5, 0) == 0.0625 and sign_test_p(2, 0) == 0.5

## Exercise 21.4: Count the paired gains and losses

**Contract.** `intervention_effect(baseline, changed_data, *, case_ids) -> InterventionSummary`

Each row in `baseline` and `changed_data` is a dictionary with `case_id`, `state_id`, `reset_seed`, `task_id`, `success`, and sometimes `prompt`. `case_ids` is the list of cases that you chose before the run, in report order. The supplied function `intervention_case_rows` makes rows from saved evaluation files. It keeps failed and error rows, and it checks the seed rule.

**Rules.**

1. `case_ids` must be nonempty and unique. Otherwise raise `ValueError`.
2. Each run must contain every selected case **exactly once**, and no other case. Otherwise raise `ValueError`.
3. For each case, `state_id`, `reset_seed`, `task_id`, and `prompt` must be equal in both runs. If a field is in one row only, that is also an error.
4. Only `success is True` counts as a success.

**Output.** `cases` (count), `baseline_successes`, `changed_data_successes`, `gains` and `losses` (lists of case IDs in `case_ids` order), and `failed_attempts_included=True`.

<details><summary>Hint</summary>

Build `{row["case_id"]: row}` for each run, and compare its size with the number of rows to find duplicates. Compare `set(keys)` with `set(case_ids)`. Loop over `case_ids` to keep the order.

</details>

In [ ]:
def intervention_effect(
    baseline: Sequence[InterventionCase],
    changed_data: Sequence[InterventionCase],
    *,
    case_ids: Sequence[str | int],
) -> InterventionSummary:
    wanted = list(case_ids)
    if not wanted or len(wanted) != len(set(wanted)):
        raise ValueError("case_ids must be nonempty and unique")
    runs: dict[str, dict[str | int, InterventionCase]] = {}
    for label, rows in (("baseline", baseline), ("changed", changed_data)):
        keyed = {row["case_id"]: row for row in rows}
        if len(keyed) != len(rows) or set(keyed) != set(wanted):
            raise ValueError(f"{label} run must contain each selected case exactly once")
        runs[label] = keyed
    for key in wanted:
        before, after = runs["baseline"][key], runs["changed"][key]
        for field in ("state_id", "reset_seed", "task_id", "prompt"):
            if (field in before) != (field in after) or before.get(field) != after.get(field):
                raise ValueError(f"paired rows differ in {field} for case {key}")
    before_ok = {key: runs["baseline"][key]["success"] is True for key in wanted}
    after_ok = {key: runs["changed"][key]["success"] is True for key in wanted}
    return {
        "cases": len(wanted),
        "baseline_successes": sum(before_ok.values()),
        "changed_data_successes": sum(after_ok.values()),
        "gains": [key for key in wanted if after_ok[key] and not before_ok[key]],
        "losses": [key for key in wanted if before_ok[key] and not after_ok[key]],
        "failed_attempts_included": True,
    }

### Check 21.4

In [ ]:
WORKED_STATES = development_cases("intervention")
WORKED_BASELINE = [False, True, False, False, True]
WORKED_CHANGED = [True, True, False, False, False]
worked_ids = [case_id(state, 20260930 + state) for state in WORKED_STATES]


def worked_rows(outcomes: list[bool]) -> list[InterventionCase]:
    return [
        {
            "case_id": name,
            "state_id": state,
            "reset_seed": 20260930 + state,
            "task_id": "bowl_plate",
            "success": outcome,
        }
        for name, state, outcome in zip(worked_ids, WORKED_STATES, outcomes, strict=True)
    ]


effect = intervention_effect(
    worked_rows(WORKED_BASELINE), worked_rows(WORKED_CHANGED), case_ids=worked_ids
)
assert effect["gains"] == [worked_ids[0]] and effect["losses"] == [worked_ids[4]], effect
assert effect["baseline_successes"] == 2 and effect["changed_data_successes"] == 2, effect
assert effect["failed_attempts_included"] is True and effect["cases"] == 5

shuffled = list(reversed(worked_rows(WORKED_CHANGED)))
assert intervention_effect(worked_rows(WORKED_BASELINE), shuffled, case_ids=worked_ids)[
    "gains"
] == [worked_ids[0]]

wrong_seed = worked_rows(WORKED_CHANGED)
wrong_seed[2] = {**wrong_seed[2], "reset_seed": 1}
for baseline_rows, changed_rows in (
    (worked_rows(WORKED_BASELINE), wrong_seed),
    (worked_rows(WORKED_BASELINE), worked_rows(WORKED_CHANGED)[:4]),
    (worked_rows(WORKED_BASELINE) * 2, worked_rows(WORKED_CHANGED) * 2),
):
    try:
        intervention_effect(baseline_rows, changed_rows, case_ids=worked_ids)
    except ValueError:
        continue
    raise AssertionError("Mismatched, missing, or repeated cases must raise ValueError")
print("21.4 passed:", effect)
print("sign-test p for these counts:", sign_test_p(len(effect["gains"]), len(effect["losses"])))

## 7. The full experiment

The cells below use real assets: the starting ACT checkpoint `act-bowl-plate-probe-200`, the training and validation data, the LIBERO simulator, and the GPU. They download nothing. The first group checks every file it needs and lists each missing file with the command that creates it. Run the cell again after you fix the list.

Each experiment group finishes before the next group starts, and a group uses only the output of earlier groups. Run them in a fresh kernel, in order. The groups use your functions from Exercises 21.1–21.4.

| Group | Kind | What it does |
| --- | --- | --- |
| `data-quality-setup` | setup | Checks the local files and the GPU. Makes a new run directory. Hashes the starting checkpoint. Saves the identity of the run: model, data split, states, seeds, updates. |
| `data-quality-collect` | data | Lets the starting policy attempt states 0–9 until it has five accepted recordings. Writes the manifest. |
| `data-quality-diagnose` | display | Applies your alignment, batch, and quality functions to every real recording. Replays each recording in the simulator. Shows what a tick defect does. |
| `data-quality-train` | training | Trains two arms from the same weights: baseline (original data) and intervention (original plus recordings). 300 updates each, with a deadline. |
| `data-quality-reload` | evaluation | Reloads both saved checkpoints from disk and checks that they give the held-out loss that they had in memory. |
| `data-quality-evaluate` | evaluation | Runs the starting checkpoint and both reloaded arms on states 15–19. Counts paired gains and losses between the two arms. |
| `data-quality-display` | display | Writes the paired summary, the time table, and the plot. |

**Budget.** Target: at most 7200 s in total on one 48 GB GPU. Each training arm stops at a deadline of `1800` s. If the deadline cuts an arm short of 300 updates, the cell raises an error, because a shorter arm would break the match. Each cell prints its own elapsed time. This notebook states no measured time, because it has no run to report.

**What the saved data prove.** 300 updates with batch size 8 are real optimizer updates, but they continue a policy that has already trained. About `N_new / (N_original + N_new)` of the training samples come from the new recordings, and the experiment prints the fraction that was really drawn. The result of five cases is a pilot signal, as section 6 shows.

In [ ]:
require_assets(
    WORKSPACE,
    {
        ".local-data/runs/act-bowl-plate-probe-200/checkpoint-000200/model.safetensors": "the starting ACT checkpoint; run `just references-install` (docs/initial-setup.md)",
        ".local-data/hub/lerobot-libero": "the training and validation data; run `just data-training`",
        ".local-data/libero-assets": "the simulator assets; run `just data-demo`",
        "configs/data/stats/bowl_plate.json": "the training statistics; restore configs/ from the repository",
    },
)
GPU_LIMIT_GB = limit_gpu_memory(48.0)
OUT_DIR = new_run_dir(DATA / "course-results", "data-quality")
BASE_CHECKPOINT = DATA / "runs/act-bowl-plate-probe-200/checkpoint-000200"
BASE_FILES_SHA256 = {
    path.name: sha256_file(path) for path in sorted(BASE_CHECKPOINT.iterdir()) if path.is_file()
}
frozen_recipe = yaml.safe_load((CONFIGS / "experiments/comparison.yaml").read_text())["policies"][
    "act"
]
if BASE_FILES_SHA256["model.safetensors"] != frozen_recipe["model_sha256"]:
    raise ValueError("The starting ACT checkpoint differs from the frozen ACT recipe")
BASE_ROBOT_STEPS = frozen_recipe["robot_training_steps"]
episodes_config = yaml.safe_load((CONFIGS / "data/episodes.yaml").read_text())
task_split = episodes_config["tasks"]["bowl_plate"]
EXPLORATION_STATES = development_cases("exploration")
INTERVENTION_STATES = development_cases("intervention")
BASE_SEED = 20260930
ARM_UPDATES = 300
ARM_SEED = 20261001
ARM_BATCH_SIZE = 8
ARM_MAX_SECONDS = 1800.0
write_run_identity(
    OUT_DIR,
    task_alias="bowl_plate",
    start_checkpoint=str(BASE_CHECKPOINT),
    start_files_sha256=BASE_FILES_SHA256,
    start_robot_steps=BASE_ROBOT_STEPS,
    dataset_revision=episodes_config["dataset_revision"],
    train_split_sha256=task_split["split_sha256"],
    exploration_states=EXPLORATION_STATES,
    intervention_states=INTERVENTION_STATES,
    reset_seed_rule="20260930 + state_id",
    arm_updates=ARM_UPDATES,
    arm_seed=ARM_SEED,
    arm_batch_size=ARM_BATCH_SIZE,
    arm_max_seconds=ARM_MAX_SECONDS,
    arms={
        "baseline": "original training split",
        "intervention": "original training split + new recordings",
    },
    chunk_size=ACT_CHUNK_SIZE,
    device="cuda",
    gpu_limit_gb=GPU_LIMIT_GB,
    control_hz=LIBERO_CONTROL_HZ,
    source_storage_fps=SOURCE_STORAGE_FPS,
)
print("run directory:", OUT_DIR)
print("starting model sha256:", BASE_FILES_SHA256["model.safetensors"])

In [ ]:
collection = collect_reference_recovery(
    checkpoint=BASE_CHECKPOINT,
    configs_root=CONFIGS,
    data_root=DATA,
    output_dir=OUT_DIR / "recordings",
    task_alias="bowl_plate",
    candidate_ids=EXPLORATION_STATES,
    target_successes=5,
)
MANIFEST_PATH = OUT_DIR / "recordings" / "manifest.json"
if not set(INTERVENTION_STATES) <= set(collection["heldout_development_state_ids"]):
    raise ValueError("Intervention states must stay held out from the recordings")
print("attempted states:", collection["attempted_state_ids"])
print(
    "accepted:",
    len(collection["recordings"]),
    "| rejected:",
    len(collection["rejected_recordings"]),
)
print("provenance limit:", collection["provenance_limit"])

In [ ]:
accepted = [RecordedEpisode.load(item["path"]) for item in collection["recordings"]]
rejected = [RecordedEpisode.load(item["path"]) for item in collection["rejected_recordings"]]
diagnosis = []
for label, episodes in (("accepted", accepted), ("rejected", rejected)):
    for episode in episodes:
        alignment = transition_alignment(episode.observation_ticks, episode.action_ticks)
        rows = aligned_transition_rows(
            episode.observations, episode.actions, episode.observation_ticks, episode.action_ticks
        )
        window, padding = action_window(
            episode.actions, start=len(rows) - 5, horizon=ACT_CHUNK_SIZE
        )
        assert len(rows) == len(episode.actions) and window.shape == (ACT_CHUNK_SIZE, 7)
        assert padding.sum() == 15, "The last row of an episode keeps five real slots"
        quality = action_quality_report(episode.actions, lower=SCHEMA.lower, upper=SCHEMA.upper)
        diagnosis.append(
            {
                "set": label,
                "episode_id": episode.episode_id,
                "alignment": alignment,
                "quality": quality,
            }
        )
        print(
            label,
            episode.episode_id,
            "| offsets:",
            alignment["offset_ticks"],
            "| ticks:",
            quality["steps"],
            "| idle:",
            round(quality["idle_fraction"], 3),
            "| saturated:",
            round(quality["saturated_fraction"], 3),
            "| gripper switches:",
            quality["gripper_switches"],
            "| max jump:",
            round(quality["max_step_change"], 3),
        )

last_rows = [(index, len(episode.actions) - 5) for index, episode in enumerate(accepted)]
real_batch = prepare_action_batch(accepted, last_rows, horizon=ACT_CHUNK_SIZE)
assert real_batch["action"].shape == (len(accepted), ACT_CHUNK_SIZE, 7)
assert (real_batch["action_is_pad"].sum(axis=1) == 15).all(), (
    "Your batch pads each episode at its own end"
)
print(
    "batch of accepted recordings:",
    real_batch["action"].shape,
    "| episodes:",
    real_batch["episode_ids"],
)

replay_differences = []
for episode in accepted:
    environment = open_libero_task("bowl_plate", int(episode.reset_seed) - BASE_SEED, CONFIGS, DATA)
    try:
        replayed = replay_episode(environment, episode)
    finally:
        environment.close()
    per_tick = [
        max(max_leaf_difference(saved, again).values())
        for saved, again in zip(episode.observations, replayed, strict=True)
    ]
    replay_differences.append({"episode_id": episode.episode_id, "max_difference": max(per_tick)})
    print("replay", episode.episode_id, "largest observation difference:", max(per_tick))

defective = apply_synthetic_timing_defect(accepted[0])
print(
    "synthetic defect report:",
    transition_alignment(defective.observation_ticks, defective.action_ticks),
)
try:
    prepare_action_batch([accepted[0], defective], [(0, 0)])
except ValueError as error:
    print("defective copy rejected by your batch function:", error)
else:
    raise AssertionError("The defective copy must be rejected")

(OUT_DIR / "diagnosis.json").write_text(
    json.dumps({"recordings": diagnosis, "replay": replay_differences}, indent=2, sort_keys=True)
    + "\n"
)
figure, axis = plt.subplots(figsize=(7, 3))
for episode in accepted:
    axis.plot(np.abs(episode.actions[:, :6]).max(axis=1), label=episode.episode_id[-8:])
axis.set_xlabel("tick (50 ms)")
axis.set_ylabel("largest arm command")
axis.legend(fontsize=7)
figure.tight_layout()
figure.savefig(OUT_DIR / "accepted-arm-commands.png", dpi=140)

In [ ]:
recovery_record = load_recovery_manifest(MANIFEST_PATH, CONFIGS)
recovery_data = RecoveryDataset(
    [Path(item["path"]) for item in recovery_record["recordings"]],
    task_id="bowl_plate",
    expected_reset_seeds={item["reset_seed"] for item in recovery_record["recordings"]},
    transition_rows_fn=aligned_transition_rows,
    window_fn=action_window,
)
arms = {}
for arm, extra in (("baseline", None), ("intervention", recovery_data)):
    arms[arm] = continue_act_matched(
        base_checkpoint=BASE_CHECKPOINT,
        output_dir=OUT_DIR / f"continued-{arm}",
        configs_root=CONFIGS,
        data_root=DATA,
        updates=ARM_UPDATES,
        seed=ARM_SEED,
        base_robot_steps=BASE_ROBOT_STEPS,
        extra_data=extra,
        max_seconds=ARM_MAX_SECONDS,
        batch_size=ARM_BATCH_SIZE,
    )
    record = arms[arm]
    print(
        arm,
        "| updates:",
        record["adaptation_updates"],
        "| stopped by:",
        record["stopped_by"],
        "| seconds:",
        round(record["train_seconds"], 1),
        "| peak allocated MiB:",
        round(record["peak_allocated_mib"]),
        "| new-data share of drawn samples:",
        round(record["new_sample_fraction"], 3),
        "| held-out loss before/after:",
        round(record["heldout_loss_before"], 4),
        round(record["heldout_loss_after"], 4),
    )
for key in (
    "adaptation_updates",
    "seed",
    "batch_size",
    "source_model_sha256",
    "learning_rate",
    "heldout_loss_before",
):
    assert arms["baseline"][key] == arms["intervention"][key], f"The arms differ in {key}"
assert (
    arms["baseline"]["new_sample_fraction"] == 0.0
    and arms["intervention"]["new_sample_fraction"] > 0.0
)
(OUT_DIR / "training.json").write_text(
    json.dumps(
        {
            "arms": arms,
            "recording_samples": len(recovery_data),
            "scope": "same start weights, new AdamW, seed, batch size, and update count; only the data differ",
        },
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

In [ ]:
RELOAD_SETTINGS = AdaptationSettings(
    seed=ARM_SEED,
    batch_size=ARM_BATCH_SIZE,
    max_updates=ARM_UPDATES,
    min_updates=ARM_UPDATES,
    device="cuda",
)
reload_report = {}
for arm, record in arms.items():
    saved_hash = sha256_file(Path(record["checkpoint"]) / "model.safetensors")
    reloaded_loss = checkpoint_heldout_loss(
        "act",
        record["checkpoint"],
        configs_root=CONFIGS,
        data_root=DATA,
        settings=RELOAD_SETTINGS,
    )
    difference = abs(reloaded_loss - record["heldout_loss_after"])
    if saved_hash != record["model_sha256"] or difference > 1e-4:
        raise AssertionError(
            f"{arm}: the reloaded checkpoint differs (loss difference {difference})"
        )
    reload_report[arm] = {"reloaded_heldout_loss": reloaded_loss, "difference": difference}
    print(arm, "| reloaded held-out loss:", round(reloaded_loss, 5), "| difference:", difference)
(OUT_DIR / "reload.json").write_text(json.dumps(reload_report, indent=2, sort_keys=True) + "\n")

In [ ]:
CHECKPOINTS = {
    "start": BASE_CHECKPOINT,
    "baseline": Path(arms["baseline"]["checkpoint"]),
    "intervention": Path(arms["intervention"]["checkpoint"]),
}
summaries = {}
records_by_arm = {}
for arm, checkpoint in CHECKPOINTS.items():
    summaries[arm] = evaluate_act_states(
        checkpoint=checkpoint,
        configs_root=CONFIGS,
        data_root=DATA,
        output_dir=OUT_DIR / f"eval-{arm}",
        state_ids=INTERVENTION_STATES,
    )
    records_by_arm[arm] = read_jsonl(OUT_DIR / f"eval-{arm}" / "episodes.jsonl")
expected_cases = [case_id(state, BASE_SEED + state) for state in INTERVENTION_STATES]
effect = intervention_effect(
    intervention_case_rows(records_by_arm["baseline"], base_seed=BASE_SEED),
    intervention_case_rows(records_by_arm["intervention"], base_seed=BASE_SEED),
    case_ids=expected_cases,
)
p_value = sign_test_p(len(effect["gains"]), len(effect["losses"]))
context_effect = intervention_effect(
    intervention_case_rows(records_by_arm["start"], base_seed=BASE_SEED),
    intervention_case_rows(records_by_arm["baseline"], base_seed=BASE_SEED),
    case_ids=expected_cases,
)
print("paired result (baseline arm vs intervention arm):", effect)
print("exact sign-test p:", p_value, "(five cases: this is a pilot signal)")
print("context (start vs baseline arm, the effect of extra updates alone):", context_effect)

In [ ]:
for arm, records in records_by_arm.items():
    times = episode_time_summary(records)
    for record in records:
        ticks = record["metrics"].get("steps")
        print(
            arm,
            "state",
            record["initial_state_id"],
            "| success:",
            record["success"],
            "| end:",
            record["termination_reason"],
            "| simulated s:",
            None if ticks is None else ticks / LIBERO_CONTROL_HZ,
            "| wall s:",
            record["metrics"].get("elapsed_s"),
        )
    print(
        arm,
        "mean simulated s:",
        times["mean_simulated_s"],
        "| mean wall s:",
        times["mean_wall_s"],
        "| wall / simulated:",
        times["wall_over_simulated"],
    )
(OUT_DIR / "paired-summary.json").write_text(
    json.dumps(
        {
            "effect": effect,
            "sign_test_p": p_value,
            "context_effect_extra_updates_only": context_effect,
            "summaries": summaries,
            "arms": arms,
            "reload": reload_report,
            "checkpoints": {arm: str(path) for arm, path in CHECKPOINTS.items()},
        },
        indent=2,
        sort_keys=True,
    )
    + "\n"
)
figure, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].bar(list(CHECKPOINTS), [summary["successes"] for summary in summaries.values()])
axes[0].set_ylabel(f"successes of {len(INTERVENTION_STATES)} cases")
axes[1].bar(list(arms), [arms[arm]["heldout_loss_after"] for arm in arms])
axes[1].axhline(arms["baseline"]["heldout_loss_before"], color="black", linestyle="--")
axes[1].set_ylabel("held-out loss; dashed: before")
figure.tight_layout()
figure.savefig(OUT_DIR / "paired-summary.png", dpi=140)

## 8. Read your result

Write short answers in your own notes. Use only numbers that your run printed.

1. **Gains and losses.** Compare the two arms: the baseline arm (300 updates on the original data) and the intervention arm (300 updates on the original data plus the recordings). How many cases changed from failure to success? How many changed from success to failure? Did a case that you hoped to fix stay a failure? Then read the context line: how did the start checkpoint compare with the baseline arm? That difference is what extra updates do without new data.
2. **Evidence.** What is the exact sign-test value? Five cases cannot give `p < 0.05`. Say what the result *can* tell you, and what it cannot.
3. **Failures.** For each failed attempt, read `termination_reason` and `simulated s`. Is it a timeout (300 ticks, 15 s), a policy error, or an infrastructure error? Do the failed cases have something in common?
4. **Data.** Open `diagnosis.json`. Did the accepted and rejected recordings differ in idle time, gripper switches, or largest jump? Did replay reproduce the saved observations? A replay difference that is not zero means the simulator is not an exact copy of the recording run. Report it; do not hide it.
5. **Provenance.** The data came from the policy itself. What bias can this add? What would a human demonstration add?
6. **Control.** Both arms had the same updates, seed, batch size, and starting weights. Which differences remain besides the data? (Hint: look at `new_sample_fraction` and at how the batches are drawn from a longer list.) Why did the held-out loss of the baseline arm also move?
7. **Time.** Compare simulated seconds (ticks divided by 20) with wall seconds for the same episodes. Which one is the robot's time, and which one is your waiting time?

Keep the run directory with your notes. A failed run is a result. It stays in the count.

## 9. Optional extensions

* **Human data.** Record 5–10 keyboard demonstrations on states 0–9 with [the recording guide](../../docs/recording.md). Build a manifest with `build_keyboard_manifest` and run the training group again with the same states, seed rule, and update count. Compare the paired result with the policy-generated result. Change only the data source.
* **Dose.** Run the training group with 100 and with 500 updates, in two new run directories. Keep the other values fixed, and keep a matched baseline arm for each. Do more updates help, or does the policy forget its earlier skill?
* **Add the rejected recordings.** Rejected attempts are real failures. Think about what training on them would teach, then test it in a new run. Do not mix this with the first intervention.
* **Defect training.** Use `apply_synthetic_timing_defect` on a copy of a recording, and train without the check. Compare it with the clean run. Do this only for study, and never with the intervention states.

## 10. What you learned

* A recording has `N` actions and `N + 1` observations. Action `i` belongs to observation `i`. Check the tick labels before you build rows.
* The simulator controls at 20 Hz. The 10 FPS value in the source dataset is a storage label. Do not repeat or skip actions because of it.
* Measure idle time, limits, gripper switches, and jumps before you train.
* A controlled intervention changes one thing, keeps the cases paired, and counts every attempt. A baseline must get the same updates, seed, and starting weights as the intervention, or extra compute is mixed with the data change. With five cases, treat the result as a pilot.

[Next: Lesson 22](../22-comparison/22-comparison.ipynb) · [Course index](../index.ipynb)